# 02 - Embeddings

**Goal:** turn each chunk of text into a list of numbers (a *vector*) that represents its **meaning**.

**Key idea:** texts with similar meaning get vectors that are close together, even if they use different words.
This is what lets us search by meaning instead of by exact keywords.

**Steps**
1. Load the embedding model
2. See what an embedding looks like
3. Measure similarity between sentences
4. Embed all our chunks
5. Save the embeddings

In [ ]:
# Install the libraries (run once). The first model download is about 90 MB.
%pip install sentence-transformers numpy

## Step 1 - Load the embedding model

In [ ]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# A small, fast model that runs on a normal laptop CPU - no API key needed
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded")

## Step 2 - What does an embedding look like?

In [ ]:
sentences = [
    "The dog is playing in the park",
    "A puppy runs on the grass",
    "Tax law changed this year",
]

# encode() turns each sentence into a vector
vectors = model.encode(sentences)

print("Shape:", vectors.shape)          # (3 sentences, 384 numbers each)
print("\nFirst 8 numbers of sentence 1:")
print(vectors[0][:8])

# TEACHING POINT: each sentence is now just 384 numbers.
# Nobody can read them, but the computer can compare them.

## Step 3 - Measure similarity

We use **cosine similarity**: a score where 1 = same meaning, 0 = unrelated.

Trick: if we *normalize* the vectors (make their length 1), cosine similarity is just a dot product.

In [ ]:
# normalize_embeddings=True makes every vector length 1
vectors = model.encode(sentences, normalize_embeddings=True)

# A matrix of dot products = similarity of every sentence with every other sentence
similarity = vectors @ vectors.T

print("Similarity matrix (rows and columns follow the sentence order):\n")
print(np.round(similarity, 2))

print("\ndog vs puppy :", round(float(similarity[0][1]), 2))
print("dog vs taxes :", round(float(similarity[0][2]), 2))

# EXPECTED: dog vs puppy is high (they share meaning, NOT words),
#           dog vs taxes is low.

Try your own sentences. This is the best way for students to build a feel for embeddings.

In [ ]:
def compare(a, b):
    # Return the similarity score between two sentences
    va, vb = model.encode([a, b], normalize_embeddings=True)
    return round(float(va @ vb), 2)

print(compare("How do I train a neural network?", "Steps for fitting a deep learning model"))
print(compare("How do I train a neural network?", "Best recipe for chapati"))

# EXERCISE: write 3 pairs of your own and guess the score BEFORE you run it.

## Step 4 - Embed all our chunks

Load the chunks saved by notebook 01, then turn every chunk into a vector.

In [ ]:
PROCESSED_DIR = Path("../data/processed")

with open(PROCESSED_DIR / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

texts = [c["text"] for c in chunks]    # we only embed the text

embeddings = model.encode(
    texts,
    batch_size=32,                 # process 32 chunks at a time
    show_progress_bar=True,
    normalize_embeddings=True,     # so a dot product = cosine similarity later
)

print("Embeddings shape:", embeddings.shape)   # (number of chunks, 384)

## Step 5 - Save the embeddings

Embedding is the slow step. We save the result so we never have to repeat it.

In [ ]:
np.save(PROCESSED_DIR / "embeddings.npy", embeddings)
print("Saved embeddings to", PROCESSED_DIR / "embeddings.npy")

# IMPORTANT: row 0 of embeddings belongs to chunks[0], row 1 to chunks[1], and so on.
# The order must never change, or the search will return the wrong text.

## Exercises for students
1. Print the shape of `embeddings`. What does each number in the shape mean?
2. Find the two chunks that are most similar to each other (hint: `embeddings @ embeddings.T`).
3. Try another model, for example `all-mpnet-base-v2`. Is it slower? Are scores different?
4. Why do we normalize the vectors?

**Next:** `03_retrieval.ipynb` - using these vectors to find the right chunks for a question.